A better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

2026-01-23 15:38:05.432837: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-23 15:38:05.435859: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Ground truth creation

We will use parameter estimates from by_cell_type models as we expect these to be the most accurate.

In [4]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [5]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")

In [6]:
primordial.compute_model_qc()

In [7]:
import pandas as pd

In [8]:
vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
vals=pd.concat(vals)
vals=vals.groupby(["cre_id","cell_type"]).mean().reset_index()
vals

,cre_id,cell_type,mu
0,Bend5_chr4_8168,EpiblastPrimitiveStreak,0.02805
1,Bend5_chr4_8168,ExEndodermParietal,0.034062
2,Bend5_chr4_8168,NeuroectodermBrain,0.019509
3,Bend5_chr4_8168,SurfaceEctoderm,0.019097
4,Bend5_chr4_8168,reference,0.046651
...,...,...,...
1457,ubcP,Mesoderm,11.870977
1458,ubcP,NeuroectodermBrain,11.051645
1459,ubcP,NeuroectodermRostral,10.44203
1460,ubcP,SurfaceEctoderm,14.532331


In [9]:
vals.dtypes

cre_id                     object
cell_type                  object
mu           Sparse[float64, 1.0]
dtype: object

In [10]:
vals["mu"] = vals["mu"].astype(float)

In [11]:
vals.dtypes

cre_id        object
cell_type     object
mu           float64
dtype: object

Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [12]:
#make "corresponding" inactive CREs...
mapping = {
    val: f"inactive_{i}"
    for i, val in enumerate(vals["cre_id"].unique())
}
mapping

{'Bend5_chr4_8168': 'inactive_0',
 'Bend5_chr4_8170': 'inactive_1',
 'Bend5_chr4_8172': 'inactive_2',
 'Bend5_chr4_8174': 'inactive_3',
 'Bend5_chr4_8175': 'inactive_4',
 'Bend5_chr4_8179': 'inactive_5',
 'Bend5_chr4_8192': 'inactive_6',
 'Bend5_chr4_8199': 'inactive_7',
 'Bend5_chr4_8201': 'inactive_8',
 'Btg1_chr10_9572': 'inactive_9',
 'Btg1_chr10_9578': 'inactive_10',
 'Btg1_chr10_9588': 'inactive_11',
 'Btg1_chr10_9593': 'inactive_12',
 'Btg1_chr10_9612': 'inactive_13',
 'Btg1_chr10_9613': 'inactive_14',
 'Cdk5r1_chr11_12559': 'inactive_15',
 'Cdk5r1_chr11_12562': 'inactive_16',
 'Cdk5r1_chr11_12574': 'inactive_17',
 'Cdk5r1_chr11_12575': 'inactive_18',
 'Cdk5r1_chr11_12582': 'inactive_19',
 'Cdk5r1_chr11_12590': 'inactive_20',
 'Cdk5r1_chr11_12595': 'inactive_21',
 'Cited2_chr10_1239': 'inactive_22',
 'Cited2_chr10_1246': 'inactive_23',
 'Cited2_chr10_1248': 'inactive_24',
 'Cited2_chr10_1253': 'inactive_25',
 'Cited2_chr10_1254': 'inactive_26',
 'Cited2_chr10_1267': 'inactive_27

In [13]:
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive=vals.copy().drop(columns=["mu"])
inactive["cre_id"] = inactive["cre_id"].map(mapping)
inactive["mu"]=minP
inactive

,cre_id,cell_type,mu
0,inactive_0,EpiblastPrimitiveStreak,0.019311
1,inactive_0,ExEndodermParietal,0.019311
2,inactive_0,NeuroectodermBrain,0.019311
3,inactive_0,SurfaceEctoderm,0.019311
4,inactive_0,reference,0.019311
...,...,...,...
1457,inactive_207,Mesoderm,0.019311
1458,inactive_207,NeuroectodermBrain,0.019311
1459,inactive_207,NeuroectodermRostral,0.019311
1460,inactive_207,SurfaceEctoderm,0.019311


This is 100%. Let us double to 200%...

In [14]:
# duplicated version with _b appended
inactive_b = inactive.copy()
inactive_b["cre_id"] = inactive_b["cre_id"] + "_b"

# stack them
inactive_double = pd.concat([inactive, inactive_b], ignore_index=True)
inactive_double

,cre_id,cell_type,mu
0,inactive_0,EpiblastPrimitiveStreak,0.019311
1,inactive_0,ExEndodermParietal,0.019311
2,inactive_0,NeuroectodermBrain,0.019311
3,inactive_0,SurfaceEctoderm,0.019311
4,inactive_0,reference,0.019311
...,...,...,...
2919,inactive_207_b,Mesoderm,0.019311
2920,inactive_207_b,NeuroectodermBrain,0.019311
2921,inactive_207_b,NeuroectodermRostral,0.019311
2922,inactive_207_b,SurfaceEctoderm,0.019311


Then stack with original gt...

In [15]:
final_gt=pd.concat([vals,inactive_double],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

,cre_id,cell_type,true_mean
0,Bend5_chr4_8168,EpiblastPrimitiveStreak,0.028050
1,Bend5_chr4_8168,ExEndodermParietal,0.034062
2,Bend5_chr4_8168,NeuroectodermBrain,0.019509
3,Bend5_chr4_8168,SurfaceEctoderm,0.019097
4,Bend5_chr4_8168,reference,0.046651
...,...,...,...
4381,inactive_207_b,Mesoderm,0.019311
4382,inactive_207_b,NeuroectodermBrain,0.019311
4383,inactive_207_b,NeuroectodermRostral,0.019311
4384,inactive_207_b,SurfaceEctoderm,0.019311


In [16]:
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [17]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [18]:
libraries[2]

,cre_id,mpra_bc,abundance
0,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAA,1.562188e-05
1,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAC,1.377567e-05
2,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAG,6.199831e-06
3,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAT,1.358808e-05
4,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAACA,3.538427e-06
...,...,...,...
83872,inactive_207_b,AAAAAAAAAAACCACTGGAA,4.625026e-07
83873,inactive_207_b,AAAAAAAAAAACCACTGGAC,2.230791e-05
83874,inactive_207_b,AAAAAAAAAAACCACTGGAG,1.470057e-05
83875,inactive_207_b,AAAAAAAAAAACCACTGGAT,4.463305e-06


In [19]:
final_gt.dtypes

cre_id        object
cell_type     object
true_mean    float64
dtype: object

# Creating sim

In [20]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-23",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=final_gt)

scMPRAforge: INFO: No 'state.parquet' found for 'twothird_pow_sim_2026-01-23'. Initalizing new object.


In [21]:
sim.gamut()

In [22]:
#sim.save()

In [23]:
#sim

Make the hypotheses...

In [24]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

In [25]:
sim.ground_truth.dtypes

cre_id       string[python]
cell_type    string[python]
true_mean           float64
dtype: object

In [26]:
#client.close()
#cluster.close()

scMPRAforge: INFO: A: drawn_library cols: Index(['cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: cre_id        object
mpra_bc       object
abundance    float64
dtype: object
scMPRAforge: INFO: A: cells_df cols: Index(['cell_type', 'cell_bc'], dtype='object'), types: cell_type    object
cell_bc      object
dtype: object
scMPRAforge: INFO: B: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: cell_type     object
cell_bc       object
cre_id        object
mpra_bc       object
abundance    float64
dtype: object
scMPRAforge: INFO: A: drawn_library cols: Index(['cre_id', 'mpra_bc', 'abundance'], dtype='object'), types: cre_id        object
mpra_bc       object
abundance    float64
dtype: object
scMPRAforge: INFO: A: cells_df cols: Index(['cell_type', 'cell_bc'], dtype='object'), types: cell_type    object
cell_bc      object
dtype: object
scMPRAforge: INFO: B: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', '

[debug] pickled df to: permerge_cells_df_c7aa06ec7f774565a4b369c8b8724de4.pkl
[debug] pickled df to: premerge_gt_4ebd2590fb9e4dffa3abfbdd47bf0de2.pkl
[debug] pickled df to: permerge_cells_df_4b7795679ce748afa797d8a1d6f9c914.pkl
[debug] pickled df to: premerge_gt_4a8cc13d750641d2bf3e3e2be05d88f3.pkl


scMPRAforge: INFO: C: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
dtype: object
scMPRAforge: INFO: C.5: cells_df cols: Index(['cre_id', 'cell_type', 'true_mean'], dtype='object'), types: cre_id       string[python]
cell_type    string[python]
true_mean           float64
dtype: object
scMPRAforge: INFO: C: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
dtype: object
scMPRAforge: INFO: C.5: cells_df cols: Index(['cre_id', 'cell_type', 'true_mean'], dtype='object'), types: cre_id       string[python]
cell_type    string[python]
true_mean           float64
dtype: object


[debug] pickled df to: permerge_cells_df_2f9a242d70f94267bd0550d22d11c873.pkl
[debug] pickled df to: premerge_gt_e01e1228b41440f19a2906144ed2c660.pkl
[debug] pickled df to: permerge_cells_df_ba45c8b686c740f4b4aae36e46c280ab.pkl
[debug] pickled df to: premerge_gt_0ee6c3cdca0a491299332a3d3e9ed250.pkl
[debug] pickled df to: postmerge_7df7fbb3e5844149a02d4a9c5c4d36f8.pkl
[debug] pickled df to: postmerge_31cba74ef25d475b937393398a5e3190.pkl


scMPRAforge: INFO: D: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'true_mean'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
true_mean           float64
dtype: object
scMPRAforge: INFO: D: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'true_mean'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
true_mean           float64
dtype: object
scMPRAforge: INFO: D 150107
scMPRAforge: INFO: D 150334


[debug] pickled df to: postmerge_693955e8e5754eadb5b10440e1f37606.pkl
[debug] pickled df to: postmerge_5cb74d9b6ec2409ba85fe5fba38cd4da.pkl


scMPRAforge: INFO: D: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'true_mean'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
true_mean           float64
dtype: object
scMPRAforge: INFO: D: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'true_mean'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
true_mean           float64
dtype: object
scMPRAforge: INFO: D 144995
scMPRAforge: INFO: D 151477
2026-01-23 15:38:30,105 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-9a1f61bd9309385678af65ffcc1606b6
State:     executing
Task:  <Task '_simulate_transfection_helper-9a1f61bd9309385678af65ffcc1606b6' _simulate_transfection_helper(, ...)>
Exception: "AssertionError('DataFrame contains NA values in these rows:\\n             cell_type     

[debug] pickled df to: permerge_cells_df_3db01db8a3b04b48904b4d6c5c5cafd4.pkl
[debug] pickled df to: premerge_gt_487170744b554bb8b52c89b69070ad90.pkl
[debug] pickled df to: postmerge_2e81c4e9326e4d348a17b45df0f1cbaf.pkl


scMPRAforge: INFO: D: cells_df cols: Index(['cell_type', 'cell_bc', 'cre_id', 'mpra_bc', 'true_mean'], dtype='object'), types: cell_type    string[python]
cell_bc              object
cre_id       string[python]
mpra_bc              object
true_mean           float64
dtype: object
scMPRAforge: INFO: D 145865
2026-01-23 15:38:31,385 - distributed.worker - ERROR - Compute Failed
Key:       _simulate_transfection_helper-e869418ced61dd6f816b27ee4dfbe4f5
State:     executing
Task:  <Task '_simulate_transfection_helper-e869418ced61dd6f816b27ee4dfbe4f5' _simulate_transfection_helper(, ...)>
Exception: "AssertionError('DataFrame contains NA values in these rows:\\n             cell_type               cell_bc  ...               mpra_bc true_mean\\n0       Cardiomyocytes  AAAAAAAAAAAAAAAAAAAA  ...  AAAAAAAAAAAAAGTCAGAT       NaN\\n1       Cardiomyocytes  AAAAAAAAAAAAAAAAAAAA  ...  AAAAAAAAAAAAACTCGCCT       NaN\\n3       Cardiomyocytes  AAAAAAAAAAAAAAAAAAAA  ...  AAAAAAAAAAAAGCAGGGGG       NaN\\n

In [27]:
cells_df=scm.load_df_pickle_debug("permerge_cells_df_ba45c8b686c740f4b4aae36e46c280ab.pkl")
#cells_df=scm.cast_string_keys(cells_df,["cell_type", "cre_id"])
ground_truth=scm.load_df_pickle_debug("premerge_gt_0ee6c3cdca0a491299332a3d3e9ed250.pkl")
#ground_truth=scm.cast_string_keys(ground_truth,["cell_type", "cre_id"])

[debug] loaded df from: permerge_cells_df_ba45c8b686c740f4b4aae36e46c280ab.pkl
[debug] loaded df from: premerge_gt_0ee6c3cdca0a491299332a3d3e9ed250.pkl


In [28]:
cells_df.merge(ground_truth,
                on=["cell_type","cre_id"],
                validate="many_to_one",
                how="left",
                indicator=True)

,cell_type,cell_bc,cre_id,mpra_bc,true_mean,_merge
0,Cardiomyocytes,AAAAAAAAAAAAAAAAAAAA,inactive_13_b,AAAAAAAAAAAATGAGGCGG,NaN,left_only
1,Cardiomyocytes,AAAAAAAAAAAAAAAAAAAA,inactive_163,AAAAAAAAAAAATAATAAGG,NaN,left_only
2,Cardiomyocytes,AAAAAAAAAAAAAAAAAAAA,Gata4_chr14_5776,AAAAAAAAAAAAAGTCTCGG,NaN,left_only
3,Cardiomyocytes,AAAAAAAAAAAAAAAAAAAA,Lama1_chr17_7784,AAAAAAAAAAAAATGATGAA,NaN,left_only
4,Cardiomyocytes,AAAAAAAAAAAAAAAAAAAA,inactive_20,AAAAAAAAAAAACTGCCTGG,NaN,left_only
...,...,...,...,...,...,...
780521,reference,AAAAAAAAAAAAGGGCCTTA,inactive_86_b,AAAAAAAAAAACAAGCTAAC,0.019311,both
780522,reference,AAAAAAAAAAAAGGGCCTTA,inactive_196,AAAAAAAAAAAATCCATATA,0.019311,both
780523,reference,AAAAAAAAAAAAGGGCCTTA,inactive_207_b,AAAAAAAAAAACCACGCAGT,0.019311,both
780524,reference,AAAAAAAAAAAAGGGCCTTA,inactive_61,AAAAAAAAAAAAGATAGGGG,0.019311,both


In [32]:
cells_df.sort_values(by=['cre_id', 'cell_type'])

,cell_type,cell_bc,cre_id,mpra_bc
170,Cardiomyocytes,AAAAAAAAAAAAAAAAAATC,Bend5_chr4_8168,AAAAAAAAAAAAAAAAATGC
440,Cardiomyocytes,AAAAAAAAAAAAAAAAACTG,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAG
1991,Cardiomyocytes,AAAAAAAAAAAAAAAACTGT,Bend5_chr4_8168,AAAAAAAAAAAAAAAAACGA
2265,Cardiomyocytes,AAAAAAAAAAAAAAAAGATG,Bend5_chr4_8168,AAAAAAAAAAAAAAAAACGC
9237,Cardiomyocytes,AAAAAAAAAAAAAAAGAACG,Bend5_chr4_8168,AAAAAAAAAAAAAAAAACAG
...,...,...,...,...
779437,reference,AAAAAAAAAAAAGGGCATTA,ubcP,AAAAAAAAAAAACGTGGGAT
780065,reference,AAAAAAAAAAAAGGGCCGAT,ubcP,AAAAAAAAAAAACGTGTGGA
780283,reference,AAAAAAAAAAAAGGGCCGTT,ubcP,AAAAAAAAAAAACGTGTGTC
780363,reference,AAAAAAAAAAAAGGGCCTAC,ubcP,AAAAAAAAAAAACGTGTAGC


In [33]:
ground_truth.sort_values(by=['cre_id', 'cell_type'])

,cre_id,cell_type,true_mean
0,Bend5_chr4_8168,EpiblastPrimitiveStreak,0.028050
1,Bend5_chr4_8168,ExEndodermParietal,0.034062
2,Bend5_chr4_8168,NeuroectodermBrain,0.019509
3,Bend5_chr4_8168,SurfaceEctoderm,0.019097
4,Bend5_chr4_8168,reference,0.046651
...,...,...,...
1457,ubcP,Mesoderm,11.870977
1458,ubcP,NeuroectodermBrain,11.051645
1459,ubcP,NeuroectodermRostral,10.442030
1460,ubcP,SurfaceEctoderm,14.532331


In [36]:
cells_df[['cre_id', 'cell_type']].drop_duplicates()

,cre_id,cell_type
0,inactive_13_b,Cardiomyocytes
1,inactive_163,Cardiomyocytes
2,Gata4_chr14_5776,Cardiomyocytes
3,Lama1_chr17_7784,Cardiomyocytes
4,inactive_20,Cardiomyocytes
...,...,...
636083,inactive_60,reference
636267,inactive_58,reference
636345,Col5a1_chr2_2575,reference
637686,inactive_17_b,reference


In [35]:
ground_truth[['cre_id', 'cell_type']].drop_duplicates()

,cre_id,cell_type
0,Bend5_chr4_8168,EpiblastPrimitiveStreak
1,Bend5_chr4_8168,ExEndodermParietal
2,Bend5_chr4_8168,NeuroectodermBrain
3,Bend5_chr4_8168,SurfaceEctoderm
4,Bend5_chr4_8168,reference
...,...,...
4381,inactive_207_b,Mesoderm
4382,inactive_207_b,NeuroectodermBrain
4383,inactive_207_b,NeuroectodermRostral
4384,inactive_207_b,SurfaceEctoderm
